# CIFAR-10 Transfer Learning (TensorFlow / Keras)
## Phase 2: ResNet50V2 | Phase 3: EfficientNetB0
- Loads preprocessed 32×32 data from `cifar10_preprocessed.npz`
- On-the-fly resize to 96×96 using `tf.image.resize`
- **Phase 2:** ResNet50V2 pretrained on ImageNet → Fine-tune
- **Phase 3:** EfficientNetB0 pretrained on ImageNet → Fine-tune
- Strategy: Freeze backbone → Train head → Unfreeze → Fine-tune all

> Note: TensorFlow/Keras does not include ResNet18. We use ResNet50V2 as the closest standard alternative.

In [19]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import time

if os.path.basename(os.getcwd()) == 'Model':
    os.chdir('..')
if os.path.basename(os.getcwd()) == 'TensorFlow':
    os.chdir('..')

## 1. Setup Device

In [20]:
print(f'TensorFlow version: {tf.__version__}')
print(f'GPUs available: {tf.config.list_physical_devices("GPU")}')
print(f'Built with CUDA: {tf.test.is_built_with_cuda()}')

TensorFlow version: 2.21.0
GPUs available: []
Built with CUDA: False


## 2. Load Data + Build tf.data Pipeline with On-the-fly Resize

In [21]:
data = np.load('TensorFlow/Preprocessing/cifar10_preprocessed.npz')

X_train = data['X_train'].astype(np.float32)
X_val   = data['X_val'].astype(np.float32)
X_test  = data['X_test'].astype(np.float32)
y_train = data['y_train'].astype(np.int64)
y_val   = data['y_val'].astype(np.int64)
y_test  = data['y_test'].astype(np.int64)
label_names = list(data['label_names'])

# Reverse CIFAR standardization (pretrained models expect their own preprocessing)
cifar_mean = data['channel_mean']; cifar_std = data['channel_std']
X_train = X_train * cifar_std + cifar_mean
X_val   = X_val   * cifar_std + cifar_mean
X_test  = X_test  * cifar_std + cifar_mean

# Scale back to 0-255 range for Keras preprocessing functions
X_train = np.clip(X_train * 255.0, 0, 255).astype(np.float32)
X_val   = np.clip(X_val   * 255.0, 0, 255).astype(np.float32)
X_test  = np.clip(X_test  * 255.0, 0, 255).astype(np.float32)

print(f'Loaded: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}')

Loaded: X_train=(67500, 32, 32, 3), X_val=(7500, 32, 32, 3), X_test=(10000, 32, 32, 3)


In [22]:
BATCH_SIZE = 64

def make_dataset(images, labels, preprocess_fn, shuffle=False):
    def preprocess(img, label):
        img = tf.image.resize(img, (96, 96))
        img = preprocess_fn(img)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if shuffle:
        ds = ds.shuffle(10000)
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

print(f'Batch size: {BATCH_SIZE}')

Batch size: 64


## 3. Helper Functions

In [23]:
def evaluate_model(model, test_ds, y_test, label_names):
    y_pred = model.predict(test_ds, verbose=0).argmax(axis=1)
    y_true = y_test
    test_acc = (y_pred == y_true).mean()

    print(f'Test Accuracy: {test_acc*100:.2f}%')
    print(f'\nPer-Class Accuracy:')
    print('-' * 30)
    for i, name in enumerate(label_names):
        mask = y_true == i
        if mask.sum() > 0:
            acc = (y_pred[mask] == i).mean()
            print(f'  {name:<12s}: {acc*100:.1f}%')
    print(f'\n{classification_report(y_true, y_pred, target_names=label_names)}')
    return y_pred, y_true, test_acc


def plot_results(history_dict, y_pred, y_true, test_acc, label_names, model_name, save_prefix):
    epochs_range = range(1, len(history_dict['loss']) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs_range, history_dict['loss'], 'b-', label='Train Loss', alpha=0.8)
    ax1.plot(epochs_range, history_dict['val_loss'], 'r-', label='Val Loss', alpha=0.8)
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs_range, history_dict['accuracy'], 'b-', label='Train Acc', alpha=0.8)
    ax2.plot(epochs_range, history_dict['val_accuracy'], 'r-', label='Val Acc', alpha=0.8)
    ax2.axhline(y=test_acc, color='green', linestyle='--', label=f'Test: {test_acc*100:.2f}%')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.suptitle(f'{model_name} — Test: {test_acc*100:.2f}%', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'TensorFlow/Figure/{save_prefix}_curves.png', dpi=150)
    plt.show()

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xticklabels(label_names, rotation=45, ha='right'); ax.set_yticklabels(label_names)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'{model_name} — Test: {test_acc*100:.2f}%')
    for i in range(10):
        for j in range(10):
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=9)
    plt.colorbar(im); plt.tight_layout()
    plt.savefig(f'TensorFlow/Figure/{save_prefix}_confusion.png', dpi=150)
    plt.show()


def merge_histories(h1, h2):
    merged = {}
    for key in h1:
        merged[key] = h1[key] + h2[key]
    return merged

---
# Phase 2: ResNet50V2 (Transfer Learning)

**Strategy:**
1. Load ResNet50V2 pretrained on ImageNet
2. **Freeze** all backbone layers (keep ImageNet knowledge)
3. Add new classification head: GlobalAveragePooling → Dropout → Dense(10)
4. Train only the new head for 10 epochs (fast)
5. **Unfreeze** entire model
6. Fine-tune all layers with small LR for 20 epochs

## Phase 2.1: Build ResNet50V2 + Create Datasets

In [25]:
# Build datasets with ResNet50V2 preprocessing
resnet_preprocess = keras.applications.resnet_v2.preprocess_input

train_ds_resnet = make_dataset(X_train, y_train, resnet_preprocess, shuffle=True)
val_ds_resnet   = make_dataset(X_val,   y_val,   resnet_preprocess)
test_ds_resnet  = make_dataset(X_test,  y_test,  resnet_preprocess)

# Build model
base_resnet = keras.applications.ResNet50V2(
    weights='imagenet', include_top=False, input_shape=(96, 96, 3)
)
base_resnet.trainable = False

inputs = keras.Input(shape=(96, 96, 3))
x = base_resnet(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation='softmax')(x)
resnet_model = keras.Model(inputs, outputs)

total_params = resnet_model.count_params()
trainable_params = sum(tf.keras.backend.count_params(w) for w in resnet_model.trainable_weights)
frozen_params = total_params - trainable_params
print(f'Total params:     {total_params:,}')
print(f'Frozen params:    {frozen_params:,}')
print(f'Trainable params: {trainable_params:,} (only the new head)')

Total params:     23,585,290
Frozen params:    23,564,800
Trainable params: 20,490 (only the new head)


## Phase 2.2: Train Head Only (Frozen Backbone) — 10 Epochs

In [27]:
resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

print('=== Phase 2.2: Training HEAD only (backbone frozen) ===')
start = time.time()
history_head = resnet_model.fit(
    train_ds_resnet,
    epochs=10,
    validation_data=val_ds_resnet,
    verbose=1
)
print(f'Completed in {time.time()-start:.1f}s')

resnet_model.save('TensorFlow/Model/best_resnet_head.h5')

=== Phase 2.2: Training HEAD only (backbone frozen) ===
Epoch 1/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 270s 254ms/step - accuracy: 0.6573 - loss: 1.0401 - val_accuracy: 0.6769 - val_loss: 0.9701
Epoch 2/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 313s 297ms/step - accuracy: 0.6585 - loss: 1.0343 - val_accuracy: 0.6713 - val_loss: 0.9698
Epoch 3/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1029s 976ms/step - accuracy: 0.6582 - loss: 1.0282 - val_accuracy: 0.6717 - val_loss: 0.9700
Epoch 4/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 290s 275ms/step - accuracy: 0.6611 - loss: 1.0238 - val_accuracy: 0.6737 - val_loss: 0.9678
Epoch 5/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 1009s 957ms/step - accuracy: 0.6609 - loss: 1.0260 - val_accuracy: 0.6703 - val_loss: 0.9872
Epoch 6/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 296s 281ms/step - accuracy: 0.6611 - loss: 1.0286 - val_accuracy: 0.6709 - val_loss: 0.9621
Epoch 7/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 309s 293ms/step - accuracy: 0.6619 - loss: 1.0231 - val_accuracy: 0.6775 - val_loss: 0.9635
Epoc

Completed in 8500.3s


## Phase 2.3: Unfreeze + Fine-tune All Layers — 20 Epochs

In [28]:
base_resnet.trainable = True

resnet_model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.001, momentum=0.9),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

callbacks_resnet = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'TensorFlow/Model/best_resnet_finetuned.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

print('=== Phase 2.3: Fine-tuning ALL layers ===')
start = time.time()
history_ft = resnet_model.fit(
    train_ds_resnet,
    epochs=20,
    validation_data=val_ds_resnet,
    callbacks=callbacks_resnet,
    verbose=1
)
print(f'Completed in {time.time()-start:.1f}s')

=== Phase 2.3: Fine-tuning ALL layers ===
Epoch 1/20
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 0s 12s/step - accuracy: 0.6904 - loss: 0.9384 
Epoch 1: val_accuracy improved from None to 0.85213, saving model to TensorFlow/Model/best_resnet_finetuned.h5



Epoch 1: finished saving model to TensorFlow/Model/best_resnet_finetuned.h5
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 13603s 13s/step - accuracy: 0.7745 - loss: 0.6606 - val_accuracy: 0.8521 - val_loss: 0.4169
Epoch 2/20
 248/1055 ━━━━━━━━━━━━━━━━━━━━ 4:00:09 18s/step - accuracy: 0.8722 - loss: 0.3669

KeyboardInterrupt: 

## Phase 2.4: ResNet50V2 Test Evaluation

In [ ]:
resnet_model = keras.models.load_model('TensorFlow/Model/best_resnet_finetuned.h5')

print('=== ResNet50V2 Test Results ===')
resnet_pred, resnet_true, resnet_test_acc = evaluate_model(
    resnet_model, test_ds_resnet, y_test, label_names
)

resnet_history = merge_histories(history_head.history, history_ft.history)
plot_results(resnet_history, resnet_pred, resnet_true, resnet_test_acc,
             label_names, 'ResNet50V2 Transfer Learning', 'resnet')

---
# Phase 3: EfficientNetB0 (Transfer Learning)

**Why EfficientNet?**
- More efficient than ResNet (fewer params, better accuracy)
- Uses compound scaling (depth + width + resolution)
- EfficientNet-B0 is the smallest variant — fast to train

**Same strategy:**
1. Freeze backbone → Train head (10 epochs)
2. Unfreeze → Fine-tune all (20 epochs)

## Phase 3.1: Build EfficientNetB0 + Create Datasets

In [ ]:
# Build datasets with EfficientNet preprocessing
effnet_preprocess = keras.applications.efficientnet.preprocess_input

train_ds_eff = make_dataset(X_train, y_train, effnet_preprocess, shuffle=True)
val_ds_eff   = make_dataset(X_val,   y_val,   effnet_preprocess)
test_ds_eff  = make_dataset(X_test,  y_test,  effnet_preprocess)

# Build model
base_effnet = keras.applications.EfficientNetB0(
    weights='imagenet', include_top=False, input_shape=(96, 96, 3)
)
base_effnet.trainable = False

inputs = keras.Input(shape=(96, 96, 3))
x = base_effnet(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation='softmax')(x)
effnet_model = keras.Model(inputs, outputs)

total_params_eff = effnet_model.count_params()
trainable_params_eff = sum(tf.keras.backend.count_params(w) for w in effnet_model.trainable_weights)
frozen_params_eff = total_params_eff - trainable_params_eff
print(f'Total params:     {total_params_eff:,}')
print(f'Frozen params:    {frozen_params_eff:,}')
print(f'Trainable params: {trainable_params_eff:,} (only the new head)')

## Phase 3.2: Train Head Only (Frozen Backbone) — 10 Epochs

In [ ]:
effnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

print('=== Phase 3.2: Training HEAD only (backbone frozen) ===')
start = time.time()
eff_history_head = effnet_model.fit(
    train_ds_eff,
    epochs=10,
    validation_data=val_ds_eff,
    verbose=1
)
print(f'Completed in {time.time()-start:.1f}s')

effnet_model.save('TensorFlow/Model/best_effnet_head.h5')

## Phase 3.3: Unfreeze + Fine-tune All Layers — 20 Epochs

In [ ]:
base_effnet.trainable = True

effnet_model.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.001, momentum=0.9),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

callbacks_eff = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'TensorFlow/Model/best_effnet_finetuned.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

print('=== Phase 3.3: Fine-tuning ALL layers ===')
start = time.time()
eff_history_ft = effnet_model.fit(
    train_ds_eff,
    epochs=20,
    validation_data=val_ds_eff,
    callbacks=callbacks_eff,
    verbose=1
)
print(f'Completed in {time.time()-start:.1f}s')

## Phase 3.4: EfficientNetB0 Test Evaluation

In [ ]:
effnet_model = keras.models.load_model('TensorFlow/Model/best_effnet_finetuned.h5')

print('=== EfficientNetB0 Test Results ===')
eff_pred, eff_true, eff_test_acc = evaluate_model(
    effnet_model, test_ds_eff, y_test, label_names
)

eff_history = merge_histories(eff_history_head.history, eff_history_ft.history)
plot_results(eff_history, eff_pred, eff_true, eff_test_acc,
             label_names, 'EfficientNetB0 Transfer Learning', 'efficientnet')

---
# Final Comparison: All Models

In [ ]:
print('=' * 60)
print('FINAL COMPARISON (TensorFlow)')
print('=' * 60)
print(f'{"Model":<30s} {"Test Acc":>10s}')
print('-' * 45)
print(f'{"Custom CNN (32x32)":<30s} {"(run CNN notebook)":>10s}')
print(f'{"ResNet50V2 (pretrained)":<30s} {resnet_test_acc*100:>9.2f}%')
print(f'{"EfficientNetB0 (pretrained)":<30s} {eff_test_acc*100:>9.2f}%')
print('=' * 60)

In [ ]:
models_names = ['Custom CNN\n(32×32)', 'ResNet50V2\n(pretrained)', 'EfficientNetB0\n(pretrained)']
accuracies = [0, resnet_test_acc * 100, eff_test_acc * 100]
colors = ['#4C72B0', '#DD8452', '#55A868']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(models_names, accuracies, color=colors, width=0.5, edgecolor='black', linewidth=0.5)
for bar, acc in zip(bars, accuracies):
    if acc > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{acc:.2f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')
    else:
        ax.text(bar.get_x() + bar.get_width() / 2, 5,
                'Run CNN\nnotebook', ha='center', va='bottom', fontsize=10, color='gray')
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison — CIFAR-10 (TensorFlow)', fontsize=14, fontweight='bold')
ax.set_ylim(0, 100); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('TensorFlow/Figure/model_comparison.png', dpi=150)
plt.show()

## Summary

| Phase | Model | Strategy | Epochs |
|---|---|---|---|
| 1 | Custom CNN | Train from scratch (see CNN notebook) | 100 |
| **2** | **ResNet50V2** | Freeze → Train head → Unfreeze → Fine-tune | 10 + 20 |
| **3** | **EfficientNetB0** | Freeze → Train head → Unfreeze → Fine-tune | 10 + 20 |

### Key Techniques Used
| Technique | Purpose |
|---|---|
| Transfer Learning | Reuse ImageNet features instead of learning from scratch |
| Freeze + Unfreeze | Prevent destroying pretrained weights early on |
| Keras preprocess_input | Match the normalization pretrained models were trained with |
| SGD + Momentum | Better generalization than Adam for fine-tuning |
| tf.data Pipeline | Efficient on-the-fly resize 32×32 → 224×224 |
| EarlyStopping | Prevent overfitting with patience-based stopping |
| ModelCheckpoint | Save best model weights automatically |